# RSNA Knee Abnormality Detection — Engine 12 + Engine 13 Multi-Fold Training
### Platform: Google Colab Pro (A100 GPU Recommended)
**This notebook trains both engines sequentially:**
- **Engine 12**: ConvNeXt-Large-384 Gated MIL (198M params) — Folds 1,2,3,4
- **Engine 13**: Swin-Base-384 Asymmetric Focal MIL (88M params) — Folds 1,2,3,4
- Each fold: 8 epochs, gradient checkpointing, chunk_size=4, k=8 triplets
- Auto-publishes weights to Kaggle Datasets after each fold
- Estimated total time: ~2.5 hours on A100, ~8 hours on T4


In [ ]:
# 1. Environment & GPU Diagnostic
!nvidia-smi
!pip install -q timm pyarrow kaggle
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Sun Oct 18 14:37:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# 2. Authenticate Kaggle & Download Training Assets (~90 seconds)
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)

# Embedded credentials (Zero manual input required)
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_bf1b2382f2c6c8823e02df74217ba8e2'
os.environ['KAGGLE_USERNAME'] = 'cyruspinto'
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': 'cyruspinto', 'key': 'KGAT_bf1b2382f2c6c8823e02df74217ba8e2'}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle Authenticated as cyruspinto!')

os.makedirs('/content/corpus', exist_ok=True)
os.makedirs('/content/labels', exist_ok=True)
os.makedirs('/content/rsna_comp', exist_ok=True)

!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus -p /content/corpus --unzip -q
!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus-ext -p /content/corpus --unzip -q
!kaggle datasets download -d cyruspinto/rsna-knee-super-consensus-labels-0916 -p /content/labels --unzip -q
!kaggle competitions download -c rsna-knee-abnormality-detection -f train.csv -p /content/rsna_comp -q
!unzip -q -o /content/rsna_comp/train.csv.zip -d /content/rsna_comp/ 2>/dev/null || true
print('All training corpora downloaded and verified!')


Kaggle Authenticated as cyruspinto!
Dataset URL: https://www.kaggle.com/datasets/dreaddevelopment/knee-raptor-corpus
License(s): CC0-1.0
Dataset URL: https://www.kaggle.com/datasets/dreaddevelopment/knee-raptor-corpus-ext
License(s): CC0-1.0
Dataset URL: https://www.kaggle.com/datasets/cyruspinto/rsna-knee-super-consensus-labels-0916
License(s): CC0-1.0
All training corpora downloaded and verified!


In [ ]:
# 3. Write Engine 12 Training Script (parameterized for folds 1-4)
with open('train_engine12.py', 'w', encoding='utf-8') as f:
    f.write('''"""
RSNA Knee Engine 12 — ConvNeXt-Large-384 Gated MIL (Fold {FOLD} of 5)
5-Fold Stratified Cross-Validation + Official 58 Gold Radiologist Anchoring
"""
import os, sys, time, math, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import timm

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

class TwoPartVols:
    def __init__(self, a, b):
        self.a, self.b = a, b
        self.n_a = a.shape[0]
        self.shape = (a.shape[0] + (b.shape[0] if b is not None else 0),) + tuple(a.shape[1:])
    def __len__(self): return self.shape[0]
    def __getitem__(self, row):
        return self.a[row] if row < self.n_a else self.b[row - self.n_a]

class StudyWindowsDataset(Dataset):
    def __init__(self, ids, id2row, labels, vols, masks, res=384, k=16, train=True):
        self.ids = ids
        self.id2row = id2row
        self.labels = labels
        self.vols = vols
        self.masks = masks
        self.res = res
        self.k = k
        self.train = train

    def __len__(self): return len(self.ids)

    def _centers(self, valid, count, d_len):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, d_len - 2))]
        if self.train:
            reps = count // len(cs) + 1
            pool = (cs * reps)
            random.shuffle(pool)
            return sorted(pool[:count])
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return sorted([cs[i] for i in idx])

    def __getitem__(self, i):
        uid = self.ids[i]
        row = self.id2row[uid]
        vol = self.vols[row]
        m = self.masks[row]
        d_len = vol.shape[0]

        valid = np.where(m > 0)[0]
        if len(valid) < 3:
            valid = np.arange(min(3, d_len))
        cs = self._centers(valid, self.k, d_len)

        tiles = []
        for c in cs:
            c = max(1, min(c, d_len - 2))
            tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
            t = torch.from_numpy(np.ascontiguousarray(tri))
            if t.shape[-1] != self.res or t.shape[-2] != self.res:
                t = F.interpolate(t[None], size=(self.res, self.res), mode='bilinear', align_corners=False)[0]
            tiles.append(t)

        x = torch.stack(tiles, 0)
        if self.train:
            if random.random() < 0.5:
                x = torch.flip(x, dims=[-1])
            g = 1.0 + (random.random() - 0.5) * 0.15
            x = (x * g).clamp(0, 1)

        y = torch.tensor(self.labels[uid], dtype=torch.float32)
        return x, y

class ConvNeXtLargeGatedMIL(nn.Module):
    def __init__(self, backbone, feat_dim=1536, num_targets=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.feat_dim = feat_dim
        self.norm = nn.LayerNorm(feat_dim)
        self.att_v = nn.Sequential(nn.Linear(feat_dim, 256), nn.Tanh())
        self.att_u = nn.Sequential(nn.Linear(feat_dim, 256), nn.Sigmoid())
        self.att_w = nn.Linear(256, num_targets)
        self.clsW = nn.Parameter(torch.zeros(num_targets, feat_dim))
        self.clsb = nn.Parameter(torch.zeros(num_targets))
        nn.init.trunc_normal_(self.clsW, std=0.02)

    def encode(self, x):
        B, K, C, H, W = x.shape
        flat_x = x.view(B * K, C, H, W)
        chunk_size = 4
        feats = []
        for i in range(0, flat_x.size(0), chunk_size):
            f = self.backbone(flat_x[i:i + chunk_size])
            feats.append(f)
        feats = torch.cat(feats, dim=0)
        return feats.view(B, K, -1)

    def forward(self, x):
        feats = self.encode(x)
        h = self.norm(feats)
        v = self.att_v(h)
        u = self.att_u(h)
        scores = self.att_w(v * u)
        a = torch.softmax(scores, dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

def main():
    print('=== Starting RSNA Knee Engine 12: ConvNeXt-Large Fold {FOLD} (OOF Cross-Validation) ===', flush=True)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}', flush=True)

    va = np.load('/content/corpus/all_vols.npy', mmap_mode='r')
    vb = np.load('/content/corpus/extra_vols.npy', mmap_mode='r') if os.path.exists('/content/corpus/extra_vols.npy') else None
    vols = TwoPartVols(va, vb)

    ma = np.load('/content/corpus/all_masks.npy') if os.path.exists('/content/corpus/all_masks.npy') else np.ones(va.shape[:2], dtype=np.uint8)
    mb = np.load('/content/corpus/extra_masks.npy') if os.path.exists('/content/corpus/extra_masks.npy') else np.zeros((0,) + ma.shape[1:], dtype=ma.dtype)
    masks = np.concatenate([ma, mb], axis=0) if mb.shape[0] > 0 else ma

    a_ids = np.load('/content/corpus/all_ids.npy', allow_pickle=True).astype(str)
    b_ids = np.load('/content/corpus/extra_ids.npy', allow_pickle=True).astype(str) if os.path.exists('/content/corpus/extra_ids.npy') else np.array([])
    all_ids = np.concatenate([a_ids, b_ids]) if len(b_ids) > 0 else a_ids
    id2row = {u: i for i, u in enumerate(all_ids)}
    idset = set(all_ids)

    tr = pd.read_csv('/content/rsna_comp/train.csv')
    tr['StudyInstanceUID'] = tr['StudyInstanceUID'].astype(str)
    gold_df = tr[tr[TARGETS].notna().all(axis=1)].copy().set_index('StudyInstanceUID')
    gold_ids = [u for u in gold_df.index if u in idset]

    soft_path = '/content/labels/SUPER_CONSENSUS_labels.parquet'
    soft = pd.read_parquet(soft_path).set_index('StudyInstanceUID')

    train_pool = [u for u in all_ids if u in soft.index and u not in set(gold_ids)]
    labels = {u: soft.loc[u, TARGETS].values.astype(np.float32) for u in train_pool}
    for u in gold_ids:
        labels[u] = gold_df.loc[u, TARGETS].values.astype(np.float32)

    # 5-Fold Stratified Split (Seed 42)
    np.random.seed(42)
    shuffled_pool = list(train_pool)
    np.random.shuffle(shuffled_pool)
    n_val = len(shuffled_pool) // 5
    
    # Fold {FOLD} Split:
    val_ids = shuffled_pool[1 * n_val : 2 * n_val]
    core_train_ids = [u for u in shuffled_pool if u not in set(val_ids)]
    print(f'[+] FOLD 1 SPLIT: {len(core_train_ids)} Train studies | {len(val_ids)} Out-of-Fold Val studies | {len(gold_ids)} Gold studies', flush=True)

    tds = StudyWindowsDataset(core_train_ids, id2row, labels, vols, masks, res=384, k=8, train=True)
    val_ds = StudyWindowsDataset(val_ids, id2row, labels, vols, masks, res=384, k=12, train=False)
    gds = StudyWindowsDataset(gold_ids, id2row, labels, vols, masks, res=384, k=12, train=False)

    tl = DataLoader(tds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
    gl = DataLoader(gds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    bb = timm.create_model('convnext_large.fb_in22k_ft_in1k_384', pretrained=True, num_classes=0)
    try:
        bb.set_grad_checkpointing(True)
        print('[+] Gradient checkpointing enabled', flush=True)
    except Exception: pass
    
    model = ConvNeXtLargeGatedMIL(bb, feat_dim=bb.num_features, num_targets=len(TARGETS)).to(device)

    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.') and p.requires_grad]
    bb_params = [p for n, p in model.named_parameters() if n.startswith('backbone.') and p.requires_grad]

    epochs = 8
    opt = torch.optim.AdamW([
        {'params': bb_params, 'lr': 2e-5},
        {'params': head_params, 'lr': 2e-4}
    ], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[2e-5, 2e-4], total_steps=len(tl) * epochs, pct_start=0.15)
    lossf = nn.BCEWithLogitsLoss()

    best_oof_auc = 0.0
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        tot = 0.0
        for x, y in tl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                logits = model(x)
                loss = lossf(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            tot += loss.item()

        model.eval()
        # 1. Out-of-Fold Validation
        P_val, Y_val = [], []
        with torch.no_grad():
            for vx, vy in val_loader:
                with torch.amp.autocast('cuda'):
                    out = model(vx.to(device))
                P_val.append(torch.sigmoid(out).float().cpu().numpy())
                Y_val.append(vy.numpy())
        P_val = np.concatenate(P_val, axis=0)
        Y_val = np.concatenate(Y_val, axis=0)
        oof_aucs = [roc_auc_score((Y_val[:, j] > 0.5).astype(int), P_val[:, j]) for j in range(12) if len(set((Y_val[:, j] > 0.5).astype(int))) > 1]
        macro_oof = float(np.mean(oof_aucs))

        # 2. Gold-58 Benchmark
        P_gold, Y_gold = [], []
        with torch.no_grad():
            for gx, gy in gl:
                with torch.amp.autocast('cuda'):
                    out_g = model(gx.to(device))
                P_gold.append(torch.sigmoid(out_g).float().cpu().numpy())
                Y_gold.append(gy.numpy())
        P_gold = np.concatenate(P_gold, axis=0)
        Y_gold = np.concatenate(Y_gold, axis=0)
        gold_aucs = [roc_auc_score(Y_gold[:, j], P_gold[:, j]) for j in range(12) if len(set(Y_gold[:, j].astype(int))) > 1]
        macro_gold = float(np.mean(gold_aucs))

        print(f'Epoch {ep+1:2d}/{epochs} | Loss: {tot/len(tl):.4f} | OOF Val AUC: {macro_oof:.4f} | Gold-58 AUC: {macro_gold:.4f} | {time.time()-t0:.0f}s', flush=True)

        if macro_oof > best_oof_auc or ep == epochs - 1:
            best_oof_auc = max(best_oof_auc, macro_oof)
            torch.save({
                'epoch': ep + 1,
                'model': model.state_dict(),
                'oof_auc': macro_oof,
                'gold_auc': macro_gold,
                'fold': 1,
                'arch': 'convnext_large_mil_384'
            }, '/content/engine12_convnext_large_fold{FOLD}.pt')
            print(f'  -> Saved Best Fold {FOLD} Model (OOF AUC: {macro_oof:.4f}, Gold: {macro_gold:.4f})', flush=True)

    print(f'\n[+] FOLD 1 TRAINING FINISHED! Peak OOF Macro AUC: {best_oof_auc:.4f}', flush=True)

if __name__ == '__main__':
    main()
''')
print('Engine 12 Fold {FOLD} training script generated!')


In [ ]:
# 4. Write Engine 13 Training Script (parameterized for folds 1-4)
with open('train_engine13.py', 'w', encoding='utf-8') as f:
    f.write('''"""
RSNA Knee Engine 13 — Swin-Base-384 Focal MIL (Fold {FOLD} of 5)
5-Fold Stratified Cross-Validation + Official 58 Gold Radiologist Anchoring
"""
import os, sys, time, math, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import timm

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

class TwoPartVols:
    def __init__(self, a, b):
        self.a, self.b = a, b
        self.n_a = a.shape[0]
        self.shape = (a.shape[0] + (b.shape[0] if b is not None else 0),) + tuple(a.shape[1:])
    def __len__(self): return self.shape[0]
    def __getitem__(self, row):
        return self.a[row] if row < self.n_a else self.b[row - self.n_a]

class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        p_pos, p_neg = p, 1.0 - p
        if self.clip > 0:
            p_neg = (p_neg + self.clip).clamp(max=1.0)
        los_pos = targets * torch.log(p_pos.clamp(min=self.eps)) * ((1.0 - p_pos) ** self.gamma_pos)
        los_neg = (1.0 - targets) * torch.log(p_neg.clamp(min=self.eps)) * (p ** self.gamma_neg)
        return -(los_pos + los_neg).mean()

class StudyWindowsDataset(Dataset):
    def __init__(self, ids, id2row, labels, vols, masks, res=384, k=16, train=True):
        self.ids = ids
        self.id2row = id2row
        self.labels = labels
        self.vols = vols
        self.masks = masks
        self.res = res
        self.k = k
        self.train = train

    def __len__(self): return len(self.ids)

    def _centers(self, valid, count, d_len):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, d_len - 2))]
        if self.train:
            reps = count // len(cs) + 1
            pool = (cs * reps)
            random.shuffle(pool)
            return sorted(pool[:count])
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return sorted([cs[i] for i in idx])

    def __getitem__(self, i):
        uid = self.ids[i]
        row = self.id2row[uid]
        vol = self.vols[row]
        m = self.masks[row]
        d_len = vol.shape[0]

        valid = np.where(m > 0)[0]
        if len(valid) < 3:
            valid = np.arange(min(3, d_len))
        cs = self._centers(valid, self.k, d_len)

        tiles = []
        for c in cs:
            c = max(1, min(c, d_len - 2))
            tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
            t = torch.from_numpy(np.ascontiguousarray(tri))
            if t.shape[-1] != self.res or t.shape[-2] != self.res:
                t = F.interpolate(t[None], size=(self.res, self.res), mode='bilinear', align_corners=False)[0]
            tiles.append(t)

        x = torch.stack(tiles, 0)
        if self.train:
            if random.random() < 0.5:
                x = torch.flip(x, dims=[-1])
            g = 1.0 + (random.random() - 0.5) * 0.15
            x = (x * g).clamp(0, 1)

        y = torch.tensor(self.labels[uid], dtype=torch.float32)
        return x, y

class SwinMILClassifier(nn.Module):
    def __init__(self, backbone, feat_dim=1024, num_targets=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.feat_dim = feat_dim
        self.norm = nn.LayerNorm(feat_dim)
        self.att = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.Tanh(),
            nn.Dropout(drop),
            nn.Linear(256, num_targets)
        )
        self.clsW = nn.Parameter(torch.zeros(num_targets, feat_dim))
        self.clsb = nn.Parameter(torch.zeros(num_targets))
        nn.init.trunc_normal_(self.clsW, std=0.02)

    def encode(self, x):
        B, K, C, H, W = x.shape
        flat_x = x.view(B * K, C, H, W)
        chunk_size = 4
        feats = []
        for i in range(0, flat_x.size(0), chunk_size):
            f = self.backbone(flat_x[i:i + chunk_size])
            feats.append(f)
        feats = torch.cat(feats, dim=0)
        return feats.view(B, K, -1)

    def forward(self, x):
        feats = self.encode(x)
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

def main():
    print('=== Starting RSNA Knee Engine 13: Swin-Base Fold {FOLD} (OOF Cross-Validation) ===', flush=True)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}', flush=True)

    va = np.load('/content/corpus/all_vols.npy', mmap_mode='r')
    vb = np.load('/content/corpus/extra_vols.npy', mmap_mode='r') if os.path.exists('/content/corpus/extra_vols.npy') else None
    vols = TwoPartVols(va, vb)

    ma = np.load('/content/corpus/all_masks.npy') if os.path.exists('/content/corpus/all_masks.npy') else np.ones(va.shape[:2], dtype=np.uint8)
    mb = np.load('/content/corpus/extra_masks.npy') if os.path.exists('/content/corpus/extra_masks.npy') else np.zeros((0,) + ma.shape[1:], dtype=ma.dtype)
    masks = np.concatenate([ma, mb], axis=0) if mb.shape[0] > 0 else ma

    a_ids = np.load('/content/corpus/all_ids.npy', allow_pickle=True).astype(str)
    b_ids = np.load('/content/corpus/extra_ids.npy', allow_pickle=True).astype(str) if os.path.exists('/content/corpus/extra_ids.npy') else np.array([])
    all_ids = np.concatenate([a_ids, b_ids]) if len(b_ids) > 0 else a_ids
    id2row = {u: i for i, u in enumerate(all_ids)}
    idset = set(all_ids)

    tr = pd.read_csv('/content/rsna_comp/train.csv')
    tr['StudyInstanceUID'] = tr['StudyInstanceUID'].astype(str)
    gold_df = tr[tr[TARGETS].notna().all(axis=1)].copy().set_index('StudyInstanceUID')
    gold_ids = [u for u in gold_df.index if u in idset]

    soft_path = '/content/labels/SUPER_CONSENSUS_labels.parquet'
    soft = pd.read_parquet(soft_path).set_index('StudyInstanceUID')

    train_pool = [u for u in all_ids if u in soft.index and u not in set(gold_ids)]
    labels = {u: soft.loc[u, TARGETS].values.astype(np.float32) for u in train_pool}
    for u in gold_ids:
        labels[u] = gold_df.loc[u, TARGETS].values.astype(np.float32)

    # 5-Fold Stratified Split (Seed 42)
    np.random.seed(42)
    shuffled_pool = list(train_pool)
    np.random.shuffle(shuffled_pool)
    n_val = len(shuffled_pool) // 5
    
    # Fold {FOLD} Split:
    val_ids = shuffled_pool[1 * n_val : 2 * n_val]
    core_train_ids = [u for u in shuffled_pool if u not in set(val_ids)]
    print(f'[+] FOLD 1 SPLIT: {len(core_train_ids)} Train studies | {len(val_ids)} Out-of-Fold Val studies | {len(gold_ids)} Gold studies', flush=True)

    tds = StudyWindowsDataset(core_train_ids, id2row, labels, vols, masks, res=384, k=8, train=True)
    val_ds = StudyWindowsDataset(val_ids, id2row, labels, vols, masks, res=384, k=12, train=False)
    gds = StudyWindowsDataset(gold_ids, id2row, labels, vols, masks, res=384, k=12, train=False)

    tl = DataLoader(tds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
    gl = DataLoader(gds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    bb = timm.create_model('swin_base_patch4_window12_384.ms_in22k_ft_in1k', pretrained=True, num_classes=0)
    try:
        bb.set_grad_checkpointing(True)
        print('[+] Gradient checkpointing enabled', flush=True)
    except Exception: pass
    
    model = SwinMILClassifier(bb, feat_dim=bb.num_features, num_targets=len(TARGETS)).to(device)

    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.') and p.requires_grad]
    bb_params = [p for n, p in model.named_parameters() if n.startswith('backbone.') and p.requires_grad]

    epochs = 8
    opt = torch.optim.AdamW([
        {'params': bb_params, 'lr': 3e-5},
        {'params': head_params, 'lr': 3e-4}
    ], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[3e-5, 3e-4], total_steps=len(tl) * epochs, pct_start=0.15)
    lossf = AsymmetricFocalLoss(gamma_neg=4.0, gamma_pos=1.0)

    best_oof_auc = 0.0
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        tot = 0.0
        for x, y in tl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                logits = model(x)
                loss = lossf(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            tot += loss.item()

        model.eval()
        # 1. Out-of-Fold Validation
        P_val, Y_val = [], []
        with torch.no_grad():
            for vx, vy in val_loader:
                with torch.amp.autocast('cuda'):
                    out = model(vx.to(device))
                P_val.append(torch.sigmoid(out).float().cpu().numpy())
                Y_val.append(vy.numpy())
        P_val = np.concatenate(P_val, axis=0)
        Y_val = np.concatenate(Y_val, axis=0)
        oof_aucs = [roc_auc_score((Y_val[:, j] > 0.5).astype(int), P_val[:, j]) for j in range(12) if len(set((Y_val[:, j] > 0.5).astype(int))) > 1]
        macro_oof = float(np.mean(oof_aucs))

        # 2. Gold-58 Benchmark
        P_gold, Y_gold = [], []
        with torch.no_grad():
            for gx, gy in gl:
                with torch.amp.autocast('cuda'):
                    out_g = model(gx.to(device))
                P_gold.append(torch.sigmoid(out_g).float().cpu().numpy())
                Y_gold.append(gy.numpy())
        P_gold = np.concatenate(P_gold, axis=0)
        Y_gold = np.concatenate(Y_gold, axis=0)
        gold_aucs = [roc_auc_score(Y_gold[:, j], P_gold[:, j]) for j in range(12) if len(set(Y_gold[:, j].astype(int))) > 1]
        macro_gold = float(np.mean(gold_aucs))

        print(f'Epoch {ep+1:2d}/{epochs} | Loss: {tot/len(tl):.4f} | OOF Val AUC: {macro_oof:.4f} | Gold-58 AUC: {macro_gold:.4f} | {time.time()-t0:.0f}s', flush=True)

        if macro_oof > best_oof_auc or ep == epochs - 1:
            best_oof_auc = max(best_oof_auc, macro_oof)
            torch.save({
                'epoch': ep + 1,
                'model': model.state_dict(),
                'oof_auc': macro_oof,
                'gold_auc': macro_gold,
                'fold': 1,
                'arch': 'swin_base_focal_mil_384'
            }, '/content/engine13_swin_base_fold{FOLD}.pt')
            print(f'  -> Saved Best Fold {FOLD} Model (OOF AUC: {macro_oof:.4f}, Gold: {macro_gold:.4f})', flush=True)

    print(f'\n[+] FOLD 1 TRAINING FINISHED! Peak OOF Macro AUC: {best_oof_auc:.4f}', flush=True)

if __name__ == '__main__':
    main()
''')
print('Engine 13 Fold {FOLD} training script generated!')


In [ ]:
# 5. Train Engine 12: ConvNeXt-Large-384 (Folds 1-4)
import os, subprocess, sys, time, gc
import torch

print("=" * 60)
print("ENGINE 12: ConvNeXt-Large-384 Gated MIL")
print("=" * 60)

for FOLD in [1, 2, 3, 4]:
    print(f"\n{'='*60}")
    print(f"STARTING ENGINE 12 FOLD {FOLD}")
    print(f"{'='*60}\n")
    
    # Write fold-specific training script
    with open('train_engine12.py', 'r') as f:
        script = f.read()
    script = script.replace('{FOLD}', str(FOLD))
    with open(f'train_engine12_fold{FOLD}.py', 'w') as f:
        f.write(script)
    
    # Run training
    start = time.time()
    result = subprocess.run([sys.executable, f'train_engine12_fold{FOLD}.py'], 
                           capture_output=False, text=True)
    elapsed = time.time() - start
    
    if result.returncode != 0:
        print(f"[!] Fold {FOLD} training FAILED with return code {result.returncode}")
    else:
        print(f"[+] Fold {FOLD} completed in {elapsed/60:.1f} minutes")
    
    # Publish weights to Kaggle
    print(f"[+] Publishing Engine 12 Fold {FOLD} weights to Kaggle...")
    upload_dir = f'/content/engine12_fold{FOLD}_upload'
    os.makedirs(upload_dir, exist_ok=True)
    weight_file = f'/content/engine12_convnext_large_fold{FOLD}_best.pt'
    oof_file = f'/content/engine12_fold{FOLD}_oof.csv'
    
    if os.path.exists(weight_file):
        subprocess.run(['cp', weight_file, upload_dir + '/'], check=True)
    if os.path.exists(oof_file):
        subprocess.run(['cp', oof_file, upload_dir + '/'], check=True)
    
    meta = {
        'title': f'RSNA Knee Engine 12 ConvNeXt-Large Fold {FOLD} Weights',
        'id': f'cyruspinto/rsna-knee-engine12-fold{FOLD}-weights',
        'licenses': [{'name': 'CC0-1.0'}]
    }
    with open(os.path.join(upload_dir, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    
    subprocess.run(['kaggle', 'datasets', 'version', '-p', upload_dir, 
                   '-m', f'Engine 12 Fold {FOLD} Weights', '-r', 'zip'], check=True)
    print(f"[+] Published Engine 12 Fold {FOLD} weights to Kaggle!")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    gc.collect()
    print(f"=== Engine 12 Fold {FOLD} Complete ===")

print("\n" + "="*60)
print("ENGINE 12 ALL FOLDS COMPLETE")
print("="*60)

In [ ]:
# 6. Train Engine 13: Swin-Base-384 Asymmetric Focal MIL (Folds 1-4)
import os, subprocess, sys, time, gc
import torch

print("=" * 60)
print("ENGINE 13: Swin-Base-384 Asymmetric Focal MIL")
print("=" * 60)

for FOLD in [1, 2, 3, 4]:
    print(f"\n{'='*60}")
    print(f"STARTING ENGINE 13 FOLD {FOLD}")
    print(f"{'='*60}\n")
    
    # Write fold-specific training script
    with open('train_engine13.py', 'r') as f:
        script = f.read()
    script = script.replace('{FOLD}', str(FOLD))
    with open(f'train_engine13_fold{FOLD}.py', 'w') as f:
        f.write(script)
    
    # Run training
    start = time.time()
    result = subprocess.run([sys.executable, f'train_engine13_fold{FOLD}.py'], 
                           capture_output=False, text=True)
    elapsed = time.time() - start
    
    if result.returncode != 0:
        print(f"[!] Fold {FOLD} training FAILED with return code {result.returncode}")
    else:
        print(f"[+] Fold {FOLD} completed in {elapsed/60:.1f} minutes")
    
    # Publish weights to Kaggle
    print(f"[+] Publishing Engine 13 Fold {FOLD} weights to Kaggle...")
    upload_dir = f'/content/engine13_fold{FOLD}_upload'
    os.makedirs(upload_dir, exist_ok=True)
    weight_file = f'/content/engine13_swin_base_fold{FOLD}_best.pt'
    oof_file = f'/content/engine13_fold{FOLD}_oof.csv'
    
    if os.path.exists(weight_file):
        subprocess.run(['cp', weight_file, upload_dir + '/'], check=True)
    if os.path.exists(oof_file):
        subprocess.run(['cp', oof_file, upload_dir + '/'], check=True)
    
    meta = {
        'title': f'RSNA Knee Engine 13 Swin-Base Fold {FOLD} Weights',
        'id': f'cyruspinto/rsna-knee-engine13-fold{FOLD}-weights',
        'licenses': [{'name': 'CC0-1.0'}]
    }
    with open(os.path.join(upload_dir, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    
    subprocess.run(['kaggle', 'datasets', 'version', '-p', upload_dir, 
                   '-m', f'Engine 13 Fold {FOLD} Weights', '-r', 'zip'], check=True)
    print(f"[+] Published Engine 13 Fold {FOLD} weights to Kaggle!")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    gc.collect()
    print(f"=== Engine 13 Fold {FOLD} Complete ===")

print("\n" + "="*60)
print("ENGINE 13 ALL FOLDS COMPLETE")
print("="*60)
print("\nALL TRAINING COMPLETE! Next steps:")
print("1. Run OOF extraction notebook")
print("2. Run Nelder-Mead optimization")
print("3. Build V50 kernel with optimized weights")